# Notebook 3 — `3_preprocessing_consolidated`

## Overview

**Purpose:** Consolidate BPS socioeconomic indicators into a unified regency-level panel covering 2016–2025. The notebook processes macroeconomic and annual socioeconomic data, harmonises the variables, handles missing observations, and exports the final analysis-ready dataset.

## Data requirements

Download the following data before running the notebook. Raw data are not redistributed in this repository in accordance with the providers' terms of use.

| Dataset | Source | Required files | Directory |
|---|---|---|---|
| GRDP (PDRB ADHK, 17 sectors) | [BPS](https://www.bps.go.id/) or data request via [BPS PST](https://pst.bps.go.id/) | `PDRB ADHK Kabupaten-Kota di Indonesia Tahun 2016-2025.xlsx` | `data/raw/BPS/BPS_Macro/` |
| Population | BPS or data request via BPS PST | `Jumlah penduduk menurut KabKota di Indonesia, 2016-2025.xlsx` | `data/raw/BPS/BPS_Macro/` |
| Unemployment rate (TPT) | BPS or data request via BPS PST | `TPT Kabkota 2016-2025.xlsx` | `data/raw/BPS/BPS_Macro/` |
| Poverty rate (P0) | BPS | Ten annual files covering 2016–2025 | `data/raw/BPS/P0/` |
| Mean years of schooling (RLS) | BPS | Ten annual files covering 2016–2025 | `data/raw/BPS/RLS/` |
| Missing-data reference | BPS data compilation | `missing_data.xlsx`, covering specified 2016 and sporadic Papua gaps | `data/raw/BPS/BPS_Macro/` |

## Workflow

| Step | Description |
|---|---|
| 1 | Process GRDP at constant prices (PDRB ADHK) across 17 economic sectors |
| 2 | Process population data from three source sheets covering 2016–2019, 2020–2023, and 2024–2025 |
| 3 | Process the unemployment rate (TPT) |
| 4 | Merge the macroeconomic indicators into a unified BPS panel |
| 5 | Process the ten annual poverty-rate (P0) files and reshape the data into a long-format panel |
| 6 | Process the ten annual mean-years-of-schooling (RLS) files |
| 7 | Compile the complete panel by combining sectoral GRDP, poverty, and RLS data |
| 8 | Handle missing observations using the specified imputation procedures and record each imputed cell in an audit log |
| 9 | Validate and export the final panel |

## Output

`data/processed/3_bps_full_panel_complete.xlsx`

The workbook contains three sheets.

| Sheet | Description |
|---|---|
| `Panel` | Consolidated regency-year socioeconomic data, including sectoral indicators |
| `Imputation_Log` | Audit trail documenting each imputed cell |
| `Data_Dictionary` | Variable definitions and descriptions |

## Important note on missing data

Approximately 10\% of socioeconomic outcome observations are imputed. The imputation procedures address specified gaps, including missing 2016 unemployment and growth values and sporadic gaps in Papua. Each imputed cell is documented in the `Imputation_Log` sheet to support transparency and reproducibility.

## STEP 1 — Imports, config, helper functions

In [5]:
# ════════════════════════════════════════════════════════════════════════════
# CONSOLIDATED IMPORTS
# ════════════════════════════════════════════════════════════════════════════

import os
import re
import glob
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ════════════════════════════════════════════════════════════════════════════
# CONFIG — paths and parameters
# ════════════════════════════════════════════════════════════════════════════

# Adjust DATA_DIR if your folder layout differs
DATA_DIR = "../data/raw/BPS"
BPS_DIR  = os.path.join(DATA_DIR, "BPS_Macro")
P0_DIR   = os.path.join(BPS_DIR,  "Po_2016_2025")
RLS_DIR  = os.path.join(BPS_DIR,  "RLS_2016_2025")

MISS_FILE   = os.path.join(BPS_DIR, "missing_data.xlsx")

OUTPUT_PATH = "../data/processed/3_bps_full_panel_complete.xlsx"

YEARS    = list(range(2016, 2026))
year_cols = [str(y) for y in YEARS]

# 17 GRDP sectors + total
SECTORS = [
    "A Pertanian, Kehutanan, dan Perikanan",
    "B Pertambangan dan Penggalian",
    "C Industri Pengolahan",
    "D Pengadaan Listrik dan Gas",
    "E Pengadaan Air; Pengelolaan Sampah, Limbah, dan Daur Ulang",
    "F Konstruksi",
    "G Perdagangan Besar dan Eceran; Reparasi Mobil dan Sepeda Motor",
    "H Transportasi dan Pergudangan",
    "I Penyediaan Akomodasi dan Makan Minum",
    "J Informasi dan Komunikasi",
    "K Jasa Keuangan dan Asuransi",
    "L Real Estat",
    "M,N Jasa Perusahaan",
    "O Administrasi Pemerintahan, Pertahanan, dan Jaminan Sosial Wajib",
    "P Jasa Pendidikan",
    "Q Jasa Kesehatan dan Kegiatan Sosial",
    "R,S,T,U Jasa Lainnya",
    "Produk Domestik Bruto",
]

# Short names for sectors (column-friendly)
SECTOR_SHORT = {
    "A Pertanian, Kehutanan, dan Perikanan"                            : "grdp_A_agriculture",
    "B Pertambangan dan Penggalian"                                    : "grdp_B_mining",
    "C Industri Pengolahan"                                            : "grdp_C_manufacturing",
    "D Pengadaan Listrik dan Gas"                                      : "grdp_D_electricity",
    "E Pengadaan Air; Pengelolaan Sampah, Limbah, dan Daur Ulang"      : "grdp_E_water",
    "F Konstruksi"                                                     : "grdp_F_construction",
    "G Perdagangan Besar dan Eceran; Reparasi Mobil dan Sepeda Motor"  : "grdp_G_trade",
    "H Transportasi dan Pergudangan"                                   : "grdp_H_transport",
    "I Penyediaan Akomodasi dan Makan Minum"                           : "grdp_I_accommodation",
    "J Informasi dan Komunikasi"                                       : "grdp_J_infocomm",
    "K Jasa Keuangan dan Asuransi"                                     : "grdp_K_finance",
    "L Real Estat"                                                     : "grdp_L_realestate",
    "M,N Jasa Perusahaan"                                              : "grdp_MN_business",
    "O Administrasi Pemerintahan, Pertahanan, dan Jaminan Sosial Wajib": "grdp_O_government",
    "P Jasa Pendidikan"                                                : "grdp_P_education",
    "Q Jasa Kesehatan dan Kegiatan Sosial"                             : "grdp_Q_health",
    "R,S,T,U Jasa Lainnya"                                             : "grdp_RSTU_other",
    "Produk Domestik Bruto"                                            : "grdp_total",
}

# ── Papua province remap (applied throughout)
# Old codes (91xxxxx, 94xxxxx): had 2016-2022 data
# New codes (92xxxxx, 95-97xxxxx): have 2023-2025 data
# After 2022 the Papua provinces were split — these are the code mappings
PAPUA_REMAP = {
    "9106000": "9203000",  # Kab. Sorong Selatan
    "9107000": "9202000",  # Kab. Sorong
    "9108000": "9201000",  # Kab. Raja Ampat
    "9109000": "9205000",  # Kab. Tambrauw
    "9110000": "9204000",  # Kab. Maybrat
    "9171000": "9271000",  # Kota Sorong
    "9401000": "9501000",  # Kab. Merauke
    "9402000": "9702000",  # Kab. Jayawijaya
    "9404000": "9604000",  # Kab. Nabire
    "9410000": "9605000",  # Kab. Paniai
    "9411000": "9608000",  # Kab. Puncak Jaya
    "9412000": "9601000",  # Kab. Mimika
    "9413000": "9502000",  # Kab. Boven Digoel
    "9414000": "9503000",  # Kab. Mappi
    "9415000": "9504000",  # Kab. Asmat
    "9416000": "9707000",  # Kab. Yahukimo
    "9417000": "9708000",  # Kab. Pegunungan Bintang
    "9418000": "9704000",  # Kab. Tolikara
    "9429000": "9701000",  # Kab. Nduga
    "9430000": "9602000",  # Kab. Puncak
    "9431000": "9710000",  # Kab. Lanny Jaya
    "9432000": "9706000",  # Kab. Mamberamo Tengah
    "9433000": "9603000",  # Kab. Dogiyai
    "9434000": "9709000",  # Kab. Yalimo
    "9435000": "9711000",  # Kab. Nduga
    "9436000": "9606000",  # Kab. Deiyai
    "9437000": "9607000",  # Kab. Intan Jaya
}
PAPUA_REMAP_4 = {k[:4]: v[:4] for k, v in PAPUA_REMAP.items()}

print(f"Config loaded")
print(f"  DATA_DIR : {DATA_DIR}")
print(f"  Years    : {YEARS[0]}–{YEARS[-1]}")
print(f"  Sectors  : {len(SECTORS)} (17 + total)")
print(f"  Papua remap: {len(PAPUA_REMAP)} regency code changes")


Config loaded
  DATA_DIR : ../data/raw/BPS
  Years    : 2016–2025
  Sectors  : 18 (17 + total)
  Papua remap: 27 regency code changes


In [6]:
# ════════════════════════════════════════════════════════════════════════════
# write_sheet — styled openpyxl helper used for the final export
# ════════════════════════════════════════════════════════════════════════════

def write_sheet(ws, df, title=None):
    """Write DataFrame to openpyxl worksheet with styled header."""
    HEADER_FILL  = PatternFill("solid", fgColor="1F4E79")
    HEADER_FONT  = Font(color="FFFFFF", bold=True, name="Arial", size=10)
    DATA_FONT    = Font(name="Arial", size=10)
    THIN         = Border(
        left=Side(style="thin"), right=Side(style="thin"),
        top=Side(style="thin"),  bottom=Side(style="thin"),
    )
    if title:
        ws.cell(row=1, column=1, value=title).font = Font(bold=True, name="Arial", size=11)
        start_row = 3
    else:
        start_row = 1

    for c, col in enumerate(df.columns, 1):
        cell = ws.cell(row=start_row, column=c, value=str(col))
        cell.fill      = HEADER_FILL
        cell.font      = HEADER_FONT
        cell.alignment = Alignment(horizontal="center", wrap_text=True)
        cell.border    = THIN

    for r, (_, row) in enumerate(df.iterrows(), start_row + 1):
        for c, col in enumerate(df.columns, 1):
            val  = row[col]
            val  = None if pd.isna(val) else val
            cell = ws.cell(row=r, column=c, value=val)
            cell.font   = DATA_FONT
            cell.border = THIN

    for c in range(1, len(df.columns) + 1):
        vals = [str(ws.cell(row=r2, column=c).value or "")
                for r2 in range(start_row, ws.max_row + 1)]
        width = min(max((len(v) for v in vals), default=10) + 2, 40)
        ws.column_dimensions[get_column_letter(c)].width = width

    ws.freeze_panes    = ws.cell(row=start_row + 1, column=1)
    ws.auto_filter.ref = ws.dimensions

print("✓ write_sheet helper defined")


✓ write_sheet helper defined


## STEP 2 — Process GRDP (PDRB ADHK)

Process GRDP at constant 2010 prices across the 17 economic sectors.

In [10]:
raw = pd.read_excel(
    f"{DATA_DIR}/BPS_Macro/PDRB ADHK Kabupaten-Kota di Indonesia Tahun 2016-2025.xlsx",
    header=None, dtype=str,
)
raw[0] = raw[0].astype(str).str.strip()

# ── Identify regency header rows: col[0] = 7-digit code, col[2:] all NaN
is_regency_header = (
    raw[0].astype(str).str.match(r"^\d{7}$", na=False) &
    raw.iloc[:, 2:].isnull().all(axis=1)
)

# ── Remove header rows + meta rows (keep only data rows)
# Meta rows: rows 0-6 (Tabel, Table, NaN, column headers, etc.)
data = raw[~is_regency_header].copy()
data = data[data[0].notna()]                                    # drop blank rows
data = data[data[0].astype(str).str.match(r"^\d{7}$", na=False)]  # only valid codes

# ── Rename columns
year_cols = [str(y) for y in YEARS]
data.columns = ["bps_kab_code", "sector"] + year_cols + ["_extra"] if len(data.columns) > 12 else \
               ["bps_kab_code", "sector"] + year_cols

# Keep only needed columns
cols_keep = ["bps_kab_code", "sector"] + year_cols
data = data[cols_keep].copy()

# ── Clean values
data["bps_kab_code"] = data["bps_kab_code"].astype(str).str.strip().str.zfill(7)
data["sector"]       = data["sector"].astype(str).str.strip()

# ── Keep only known sectors
data = data[data["sector"].isin(SECTORS)].copy()

# ── Coerce year columns to numeric
for yr in year_cols:
    data[yr] = pd.to_numeric(data[yr], errors="coerce")

# ── Papua province remap
# When Papua split into 6 new provinces (2022), BPS assigned new 7-digit codes
# to the same regencies. Old codes (91xxxxx, 94xxxxx) have data 2016-2022,
# new codes (92xxxxx, 95xxxxx, 96xxxxx, 97xxxxx) have data 2023-2025.
# Strategy: replace old code with new canonical code, then combine rows
# so each regency has a complete 2016-2025 series.
PAPUA_REMAP = {
    # Old code  : New code      # Regency name
    "9106000"   : "9203000",    # Kab. Sorong Selatan  -> Papua Barat Daya
    "9107000"   : "9202000",    # Kab. Sorong          -> Papua Barat Daya
    "9108000"   : "9201000",    # Kab. Raja Ampat      -> Papua Barat Daya
    "9109000"   : "9205000",    # Kab. Tambrauw        -> Papua Barat Daya
    "9110000"   : "9204000",    # Kab. Maybrat         -> Papua Barat Daya
    "9171000"   : "9271000",    # Kota Sorong          -> Papua Barat Daya
    "9401000"   : "9501000",    # Kab. Merauke         -> Papua Selatan
    "9402000"   : "9702000",    # Kab. Jayawijaya      -> Papua Pegunungan
    "9404000"   : "9604000",    # Kab. Nabire          -> Papua Tengah
    "9410000"   : "9605000",    # Kab. Paniai          -> Papua Tengah
    "9411000"   : "9608000",    # Kab. Puncak Jaya     -> Papua Tengah
    "9412000"   : "9601000",    # Kab. Mimika          -> Papua Tengah
    "9413000"   : "9502000",    # Kab. Boven Digoel    -> Papua Selatan
    "9414000"   : "9503000",    # Kab. Mappi           -> Papua Selatan
    "9415000"   : "9504000",    # Kab. Asmat           -> Papua Selatan
    "9416000"   : "9707000",    # Kab. Yahukimo        -> Papua Pegunungan
    "9417000"   : "9708000",    # Kab. Pegunungan Bintang -> Papua Pegunungan
    "9418000"   : "9704000",    # Kab. Tolikara        -> Papua Pegunungan
    "9429000"   : "9701000",    # Kab. Nduga           -> Papua Pegunungan
    "9430000"   : "9703000",    # Kab. Lanny Jaya      -> Papua Pegunungan
    "9431000"   : "9705000",    # Kab. Mamberamo Tengah -> Papua Pegunungan
    "9432000"   : "9706000",    # Kab. Yalimo          -> Papua Pegunungan
    "9433000"   : "9607000",    # Kab. Puncak          -> Papua Tengah
    "9434000"   : "9602000",    # Kab. Dogiyai         -> Papua Tengah
    "9435000"   : "9606000",    # Kab. Intan Jaya      -> Papua Tengah
    "9436000"   : "9603000",    # Kab. Deiyai          -> Papua Tengah
}

n_before = data["bps_kab_code"].nunique()
data["bps_kab_code"] = data["bps_kab_code"].replace(PAPUA_REMAP)

# Combine rows: old code had 2016-2022, new code had 2023-2025
# After remap both share the same code — merge by taking non-NaN value per year
def combine_series(group):
    """For duplicate code+sector rows, combine by taking non-NaN value per year."""
    if len(group) == 1:
        return group.iloc[0]
    result = group.iloc[0].copy()
    for col in year_cols:
        non_null = group[col].dropna()
        result[col] = non_null.iloc[0] if len(non_null) > 0 else float("nan")
    return result

data = (
    data.groupby(["bps_kab_code", "sector"], as_index=False)
    .apply(combine_series, include_groups=False)
    .reset_index(drop=True)
)

n_after = data["bps_kab_code"].nunique()
print(f"Papua remap applied:")
print(f"  Before: {n_before} regency codes")
print(f"  After : {n_after} regency codes  ({n_before - n_after} old codes merged into new)")
print(f"  Expected: 514 codes (540 - 26 remapped)")

# ── Pivot to wide: one row per regency, columns = sector × year
data_long = data.melt(
    id_vars   = ["bps_kab_code", "sector"],
    value_vars = year_cols,
    var_name  = "year",
    value_name = "value",
)
data_long["year"] = data_long["year"].astype(int)

# Add short column name
data_long["col_name"] = data_long["sector"].map(SECTOR_SHORT) + "_" + data_long["year"].astype(str)

# Pivot wide: rows = regency, cols = grdp_A_agriculture_2016 etc.
pdrb_wide = data_long.pivot_table(
    index   = "bps_kab_code",
    columns = "col_name",
    values  = "value",
    aggfunc = "first",
).reset_index()

# ── Also create long panel: bps_kab_code × year × sector
pdrb_panel = data_long[["bps_kab_code", "year", "sector", "value"]].copy()
pdrb_panel["sector_short"] = pdrb_panel["sector"].map(SECTOR_SHORT)

# ── Create GRDP growth column
grdp_total = pdrb_panel[pdrb_panel["sector"] == "Produk Domestik Bruto"][
    ["bps_kab_code", "year", "value"]
].copy().rename(columns={"value": "grdp_total_billion_idr"})

grdp_total = grdp_total.sort_values(["bps_kab_code", "year"])
grdp_total["grdp_growth_pct"] = (
    grdp_total.groupby("bps_kab_code")["grdp_total_billion_idr"]
    .pct_change() * 100
)

print(f"PDRB ADHK processed:")
print(f"  Regencies : {pdrb_panel['bps_kab_code'].nunique()}")
print(f"  Sectors   : {pdrb_panel['sector'].nunique()}")
print(f"  Panel obs : {len(grdp_total)}")


# ═════════════════ CELL 1 ═════════════════
# ═════════════════════════════════════════════════════════════════════════════

Papua remap applied:
  Before: 540 regency codes
  After : 514 regency codes  (26 old codes merged into new)
  Expected: 514 codes (540 - 26 remapped)
PDRB ADHK processed:
  Regencies : 514
  Sectors   : 18
  Panel obs : 5140


## STEP 3 — Process population

Process population from three source sheets covering 2016–2019, 2020–2023, and 2024–2025.

In [14]:
pop_frames = []

# ── Papua province remap for population (4-digit codes)
# Sheets 2016-2019 and 2020-2023 use old 4-digit codes (9106, 9107, etc.)
# Sheet 2024-2025 already uses new 4-digit codes (9201, 9202, etc.)
# Apply remap to sheets 2016-2019 and 2020-2023 only
PAPUA_REMAP_4 = {k[:4]: v[:4] for k, v in PAPUA_REMAP.items()}

# Sheet 1: 2016-2019
s1 = pd.read_excel(
    f"{DATA_DIR}/BPS_Macro/Jumlah penduduk menurut KabKota di Indonesia, 2016-2025.xlsx",
    sheet_name = "2016-2019",
    header     = None,
)
s1_data = s1[s1[0].astype(str).str.match(r"^\d{4}$", na=False)].copy()
s1_data.columns = ["kode", "name", "2016", "2017", "2018", "2019"]
s1_data["kode"] = s1_data["kode"].astype(str).str.strip().str.zfill(4)
s1_data["kode"] = s1_data["kode"].replace(PAPUA_REMAP_4)   # ← remap old → new
s1_data = s1_data.melt(
    id_vars=["kode", "name"], var_name="year", value_name="population"
)
pop_frames.append(s1_data)

# Sheet 2: 2020-2023
s2 = pd.read_excel(
    f"{DATA_DIR}/BPS_Macro/Jumlah penduduk menurut KabKota di Indonesia, 2016-2025.xlsx",
    sheet_name = "2020-2023",
    header     = None,
)
s2_data = s2[s2[0].astype(str).str.match(r"^\d{4}$", na=False)].copy()
s2_data.columns = ["kode", "name", "2020", "2021", "2022", "2023"]
s2_data["kode"] = s2_data["kode"].astype(str).str.strip().str.zfill(4)
s2_data["kode"] = s2_data["kode"].replace(PAPUA_REMAP_4)   # ← remap old → new
s2_data = s2_data.melt(
    id_vars=["kode", "name"], var_name="year", value_name="population"
)
pop_frames.append(s2_data)

# Sheet 3: 2024-2025
# Structure: col[1]=4-digit code, col[2]=name, col[3]=2024, col[4]=2025
# Already uses new codes — no remap needed
s3 = pd.read_excel(
    f"{DATA_DIR}/BPS_Macro/Jumlah penduduk menurut KabKota di Indonesia, 2016-2025.xlsx",
    sheet_name = "2024-2025",
    header     = None,
)
s3_data = s3[s3[1].astype(str).str.match(r"^\d{4}$", na=False)].copy()
s3_data = s3_data[[1, 2, 3, 4]].copy()
s3_data.columns = ["kode", "name", "2024", "2025"]
s3_data = s3_data.melt(
    id_vars=["kode", "name"], var_name="year", value_name="population"
)
pop_frames.append(s3_data)

# Combine all years
population = pd.concat(pop_frames, ignore_index=True)
population["kode"] = population["kode"].astype(str).str.strip().str.zfill(4)
population["year"] = population["year"].astype(str).str.replace("*", "", regex=False).astype(int)
population["population"] = pd.to_numeric(population["population"], errors="coerce")
population = population.rename(columns={"kode": "bps_kab_code_4", "name": "regency_name_pop"})
population = population.sort_values(["bps_kab_code_4", "year"]).reset_index(drop=True)

print(f"Population processed:")
print(f"  Regencies : {population['bps_kab_code_4'].nunique()}")
print(f"  Years     : {sorted(population['year'].unique())}")
print(f"  Panel obs : {len(population)}")
print(f"  Missing   : {population['population'].isna().sum()}")


Population processed:
  Regencies : 514
  Years     : [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
  Panel obs : 5140
  Missing   : 0


## STEP 4 — Process unemployment rate (TPT)

Process the open unemployment rate.

In [16]:
raw_tpt = pd.read_excel(
    f"{DATA_DIR}/BPS_Macro/TPT Kabkota 2016-2025.xlsx",
    header=None, dtype=str,
)

# Col[1] has "1101 Simeulue" format — extract 4-digit code
# Col[2] = 2016, Col[3] = 2017, ..., Col[11] = 2025
# Skip rows where col[1] has no 4-digit code
tpt_data = raw_tpt[
    raw_tpt[1].astype(str).str.match(r"^\d{4}\s+\S+", na=False)
].copy()

tpt_data = tpt_data.iloc[:, 1:12].copy()
tpt_data.columns = ["kode_name", "2016", "2017", "2018", "2019",
                    "2020", "2021", "2022", "2023", "2024", "2025"]

# Extract 4-digit code from "1101 Simeulue"
tpt_data["bps_kab_code_4"] = tpt_data["kode_name"].str.extract(r"^(\d{4})")
tpt_data["regency_name_tpt"] = tpt_data["kode_name"].str.replace(r"^\d{4}\s+", "", regex=True).str.strip()

tpt_year_cols = [str(y) for y in YEARS]
for yr in tpt_year_cols:
    tpt_data[yr] = pd.to_numeric(tpt_data[yr], errors="coerce")

# ── Papua province remap for TPT
# TPT file has 545 rows — both old and new codes present
# Old codes (9106, 9107, etc.): all values are 0 or NaN — empty rows
# New codes (9201, 9202, etc.): already have complete data 2017-2025
# Strategy: remap old → new, then combine rows keeping non-zero/non-NaN values
n_before_tpt = tpt_data["bps_kab_code_4"].nunique()
tpt_data["bps_kab_code_4"] = tpt_data["bps_kab_code_4"].replace(PAPUA_REMAP_4)

def combine_tpt(group):
    """Keep the row with actual data — old code rows are all 0/NaN."""
    if len(group) == 1:
        return group.iloc[0]
    result = group.iloc[0].copy()
    # Use name from new code row (non-empty name)
    non_empty_names = group["regency_name_tpt"][group["regency_name_tpt"].str.strip() != ""]
    result["regency_name_tpt"] = non_empty_names.iloc[-1] if len(non_empty_names) > 0 else ""
    for col in tpt_year_cols:
        # Prefer non-zero, non-NaN values from any row
        valid = group[col][group[col].notna() & (group[col] != 0)]
        result[col] = valid.iloc[0] if len(valid) > 0 else float("nan")
    return result

tpt_data = (
    tpt_data.groupby("bps_kab_code_4", as_index=False)
    .apply(combine_tpt, include_groups=False)
    .reset_index(drop=True)
)
n_after_tpt = tpt_data["bps_kab_code_4"].nunique()
print(f"Papua remap applied (TPT):")
print(f"  Before: {n_before_tpt} codes  →  After: {n_after_tpt} codes "
      f"({n_before_tpt - n_after_tpt} old codes merged into new)")

# 2016 is all 0 (estimasi provinsi) → replace with NaN
tpt_data["2016"] = np.nan

tpt_long = tpt_data.melt(
    id_vars    = ["bps_kab_code_4", "regency_name_tpt"],
    value_vars = tpt_year_cols,
    var_name   = "year",
    value_name = "tpt_pct",
)
tpt_long["year"] = tpt_long["year"].astype(int)
tpt_long = tpt_long.sort_values(["bps_kab_code_4", "year"]).reset_index(drop=True)

print(f"TPT processed:")
print(f"  Regencies : {tpt_long['bps_kab_code_4'].nunique()}")
print(f"  Years     : {sorted(tpt_long['year'].unique())}")
print(f"  Note      : 2016 set to NaN (province-level estimate only)")
print(f"  Panel obs : {len(tpt_long)}")

Papua remap applied (TPT):
  Before: 540 codes  →  After: 514 codes (26 old codes merged into new)
TPT processed:
  Regencies : 514
  Years     : [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
  Note      : 2016 set to NaN (province-level estimate only)
  Panel obs : 5140


## STEP 5 — Merge into unified BPS panel

Merge the GRDP, population, and unemployment series into a single long-format BPS panel.

In [18]:
# GRDP total + growth is based on 7-digit bps_kab_code
# Population and TPT use 4-digit code

# Create 4-digit version of bps_kab_code from grdp_total
grdp_total["bps_kab_code_4"] = grdp_total["bps_kab_code"].str[:4]

# Start panel from GRDP total (most complete)
panel = grdp_total.copy()

# Merge population — also pulls regency_name_pop, since the per-capita
# source that used to supply the regency name column was removed above
panel = panel.merge(
    population[["bps_kab_code_4", "year", "population", "regency_name_pop"]],
    on  = ["bps_kab_code_4", "year"],
    how = "left",
)
panel = panel.rename(columns={"regency_name_pop": "regency_name"})

# Merge TPT
panel = panel.merge(
    tpt_long[["bps_kab_code_4", "year", "tpt_pct"]],
    on  = ["bps_kab_code_4", "year"],
    how = "left",
)

# ── Final column order
final_cols = [
    "bps_kab_code",          # 7-digit BPS code
    "bps_kab_code_4",        # 4-digit BPS code
    "regency_name",          # regency name
    "year",                  # year
    "grdp_total_billion_idr",# GRDP total (billion IDR, constant 2010)
    "grdp_growth_pct",       # GRDP growth rate (%)
    "population",            # population
    "tpt_pct",               # open unemployment rate (%)
]
panel = panel[final_cols].copy()
panel = panel.sort_values(["bps_kab_code_4", "year"]).reset_index(drop=True)

# ── Summary
n_total   = len(panel)
n_reg     = panel["bps_kab_code_4"].nunique()
n_years   = panel["year"].nunique()
n_missing = panel[["grdp_total_billion_idr",
                    "population", "tpt_pct"]].isna().sum()

print(f"\nPanel summary:")
print(f"  Regencies × years  : {n_reg} × {n_years} = {n_total:,} obs")
print(f"  Years covered      : {sorted(panel['year'].unique())}")
print(f"\nMissing values per column:")
print(n_missing.to_string())


Panel summary:
  Regencies × years  : 514 × 10 = 5,140 obs
  Years covered      : [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Missing values per column:
grdp_total_billion_idr      0
population                  0
tpt_pct                   521


## STEP 6 — Process poverty rate (P0)

Read the 10 annual poverty files and consolidate them into a long-format panel.

In [21]:
# ════════════════════════════════════════════════════════════════════════════
# Process P0 poverty files — produces `poverty_long` (long-format panel)
# Replaces the old `poverty_p0_panel.xlsx` intermediate file
# ════════════════════════════════════════════════════════════════════════════

def read_p0_file(path: str, year: int) -> pd.DataFrame | None:
    """
    Read one annual P0 file.
    Keeps only rows where col[6] (bps_kab_code) is a valid number.
    Province rows (col[6] = NaN) are excluded automatically.
    """
    try:
        raw = pd.read_excel(path, header=None, dtype=str)
    except Exception as e:
        print(f"  [ERROR] {year}: {e}")
        return None

    # Keep rows where col[6] = numeric bps_kab_code
    mask = pd.to_numeric(raw[6], errors="coerce").notna()
    df   = raw[mask].copy()

    df["bps_kab_code"]  = (
        pd.to_numeric(df[6], errors="coerce")
        .astype(int).astype(str).str.zfill(4)
    )
    df["bps_kab_name"]  = df[5].astype(str).str.strip()
    df["bps_prov_name"] = df[3].astype(str).str.strip()
    df["bps_prov_code"] = pd.to_numeric(df[4], errors="coerce").astype("Int64")
    df["poverty_pct"]   = pd.to_numeric(df[1], errors="coerce")
    df["year"]          = year

    return df[[
        "bps_kab_code", "bps_kab_name", "bps_prov_name",
        "bps_prov_code", "poverty_pct", "year"
    ]].reset_index(drop=True)


# =============================================================================
# STEP 1 -- READ ALL ANNUAL FILES
# =============================================================================

print("=" * 60)
print("STEP 1 -- Read annual P0 files")
print("=" * 60)

frames        = []
missing_years = []

for year in YEARS:
    patterns = [
        f"{P0_DIR}/Persentase_Penduduk_Miskin__P0__Menurut_Kabupaten_Kota__{year}.xlsx",
        f"{P0_DIR}/Persentase Penduduk Miskin (P0) Menurut Kabupaten_Kota, {year}.xlsx",
        f"{P0_DIR}/*P0*{year}*.xlsx",
        f"{P0_DIR}/*{year}*.xlsx",
    ]

    path = None
    for pat in patterns:
        matches = glob.glob(pat)
        if matches:
            path = matches[0]
            break

    if path is None:
        print(f"  [SKIP] {year}: file not found")
        missing_years.append(year)
        continue

    df = read_p0_file(path, year)
    if df is None:
        missing_years.append(year)
        continue

    n_valid   = df["poverty_pct"].notna().sum()
    n_missing = df["poverty_pct"].isna().sum()
    frames.append(df)
    print(f"  [OK] {year}: {len(df):,} regencies "
          f"| valid={n_valid}  missing={n_missing} "
          f"| {os.path.basename(path)}")

if not frames:
    raise SystemExit("No P0 files found. Check P0_DIR path.")

if missing_years:
    print(f"\nMissing years: {missing_years}")


# =============================================================================
# STEP 2 -- COMBINE AND CLEAN
# =============================================================================

print("\n" + "=" * 60)
print("STEP 2 -- Combine and clean")
print("=" * 60)

poverty_long = pd.concat(frames, ignore_index=True)
poverty_long = poverty_long.sort_values(["bps_kab_code", "year"]).reset_index(drop=True)

n_reg   = poverty_long["bps_kab_code"].nunique()
n_years = poverty_long["year"].nunique()
n_obs   = len(poverty_long)

print(f"Panel: {n_reg} regencies x {n_years} years = {n_obs:,} obs")
print(f"Years: {sorted(poverty_long['year'].unique())}")
print(f"Missing poverty_pct: {poverty_long['poverty_pct'].isna().sum()}")

year_summary = (
    poverty_long.groupby("year")
    .agg(
        regencies = ("bps_kab_code", "nunique"),
        valid     = ("poverty_pct", lambda x: x.notna().sum()),
        missing   = ("poverty_pct", lambda x: x.isna().sum()),
        mean_p0   = ("poverty_pct", "mean"),
        min_p0    = ("poverty_pct", "min"),
        max_p0    = ("poverty_pct", "max"),
    )
    .reset_index()
)
print("\nCoverage by year:")
print(year_summary.to_string(index=False))


# =============================================================================
# STEP 3 -- WIDE FORMAT
# =============================================================================

print("\n" + "=" * 60)
print("STEP 3 -- Create poverty_wide format")
print("=" * 60)

# One consistent name per regency from most recent year
name_map = (
    poverty_long.sort_values("year", ascending=False)
    .drop_duplicates(subset="bps_kab_code")
    [["bps_kab_code", "bps_kab_name", "bps_prov_name", "bps_prov_code"]]
)

poverty_wide = poverty_long.pivot_table(
    index   = "bps_kab_code",
    columns = "year",
    values  = "poverty_pct",
    aggfunc = "first",
).reset_index()
poverty_wide.columns.name = None
poverty_wide.columns = ["bps_kab_code"] + [f"poverty_{y}" for y in poverty_wide.columns[1:]]
poverty_wide = poverty_wide.merge(name_map, on="bps_kab_code", how="left")

year_cols = [f"poverty_{y}" for y in YEARS if f"poverty_{y}" in poverty_wide.columns]
poverty_wide = poverty_wide[["bps_kab_code", "bps_kab_name", "bps_prov_name",
             "bps_prov_code"] + year_cols].copy()

print(f"Wide format: {len(poverty_wide)} regencies x {len(poverty_wide.columns)} columns")


# =============================================================================


# ── Rename to match downstream merge expectations ─────────────────────────
# bps_kab_code in this section is the 4-digit code (built from col[6] of source files).
# Rename to bps_kab_code_4 for consistency with the BPS panel from Section 5,
# which is what the merge in Section 8 joins on.
poverty_long = poverty_long.rename(columns={"bps_kab_code": "bps_kab_code_4"})
poverty_wide = poverty_wide.rename(columns={"bps_kab_code": "bps_kab_code_4"})

print(f"\nPoverty P0 processed:")
print(f"  Regencies : {poverty_long['bps_kab_code_4'].nunique() if 'poverty_long' in dir() else 'NA'}")
print(f"  Years     : {sorted(poverty_long['year'].unique()) if 'poverty_long' in dir() else 'NA'}")
print(f"  Panel obs : {len(poverty_long) if 'poverty_long' in dir() else 'NA'}")

STEP 1 -- Read annual P0 files
  [OK] 2016: 514 regencies | valid=514  missing=0 | Persentase Penduduk Miskin (P0) Menurut Kabupaten_Kota, 2016.xlsx
  [OK] 2017: 514 regencies | valid=514  missing=0 | Persentase Penduduk Miskin (P0) Menurut Kabupaten_Kota, 2017.xlsx
  [OK] 2018: 514 regencies | valid=514  missing=0 | Persentase Penduduk Miskin (P0) Menurut Kabupaten_Kota, 2018.xlsx
  [OK] 2019: 514 regencies | valid=514  missing=0 | Persentase Penduduk Miskin (P0) Menurut Kabupaten_Kota, 2019.xlsx
  [OK] 2020: 514 regencies | valid=514  missing=0 | Persentase Penduduk Miskin (P0) Menurut Kabupaten_Kota, 2020.xlsx
  [OK] 2021: 514 regencies | valid=514  missing=0 | Persentase Penduduk Miskin (P0) Menurut Kabupaten_Kota, 2021.xlsx
  [OK] 2022: 514 regencies | valid=514  missing=0 | Persentase Penduduk Miskin (P0) Menurut Kabupaten_Kota, 2022.xlsx
  [OK] 2023: 514 regencies | valid=514  missing=0 | Persentase Penduduk Miskin (P0) Menurut Kabupaten_Kota, 2023.xlsx
  [OK] 2024: 514 regencie

## STEP 7 — Process mean years of schooling (RLS)

Read the 10 annual schooling files and consolidate them into a long-format panel.

In [24]:
# ════════════════════════════════════════════════════════════════════════════
# Process RLS files — produces `rls_long` (long-format panel)
# Replaces the old `rls_panel.xlsx` intermediate file
# ════════════════════════════════════════════════════════════════════════════

def read_rls_file(path: str, year: int) -> pd.DataFrame | None:
    """
    Read one annual RLS file.
    Keeps only rows where col[6] (bps_kab_code) is a valid number.
    Province rows (col[6] = NaN) are excluded automatically.
    """
    try:
        raw = pd.read_excel(path, header=None, dtype=str)
    except Exception as e:
        print(f"  [ERROR] {year}: {e}")
        return None

    mask = pd.to_numeric(raw[6], errors="coerce").notna()
    df   = raw[mask].copy()

    df["bps_kab_code"]  = (
        pd.to_numeric(df[6], errors="coerce")
        .astype(int).astype(str).str.zfill(4)
    )
    df["bps_kab_name"]  = df[5].astype(str).str.strip()
    df["bps_prov_name"] = df[3].astype(str).str.strip()
    df["bps_prov_code"] = pd.to_numeric(df[4], errors="coerce").astype("Int64")
    df["rls_years"]     = pd.to_numeric(df[1], errors="coerce")
    df["year"]          = year

    return df[[
        "bps_kab_code", "bps_kab_name", "bps_prov_name",
        "bps_prov_code", "rls_years", "year"
    ]].reset_index(drop=True)


# =============================================================================
# STEP 1 -- READ ALL ANNUAL FILES
# =============================================================================

print("=" * 60)
print("STEP 1 -- Read annual RLS files")
print("=" * 60)

# Show what files exist in the folder
print(f"\nFiles in RLS_DIR:")
try:
    for f in sorted(os.listdir(RLS_DIR)):
        if f.endswith(".xlsx"):
            print(f"  {f}")
except FileNotFoundError:
    print(f"  FOLDER NOT FOUND: {RLS_DIR}")
    print("  Please update RLS_DIR path at the top of this script.")
    raise SystemExit("Folder not found.")
print()

frames        = []
missing_years = []

for year in YEARS:
    # Try multiple filename patterns to handle naming variations
    patterns = [
        f"{RLS_DIR}/_Metode_Baru__Rata-rata_Lama_Sekolah__{year}.xlsx",
        # Alternative with single underscore
        f"{RLS_DIR}/_Metode_Baru__Rata-rata_Lama_Sekolah_{year}.xlsx",
        # Without leading underscore
        f"{RLS_DIR}/Metode_Baru__Rata-rata_Lama_Sekolah__{year}.xlsx",
        # Wildcard fallback
        f"{RLS_DIR}/*{year}*.xlsx",
        f"{RLS_DIR}/*Lama*{year}*.xlsx",
        f"{RLS_DIR}/*RLS*{year}*.xlsx",
    ]

    path = None
    for pat in patterns:
        matches = glob.glob(pat)
        if matches:
            path = matches[0]
            break

    if path is None:
        print(f"  [SKIP] {year}: file not found")
        missing_years.append(year)
        continue

    df = read_rls_file(path, year)
    if df is None:
        missing_years.append(year)
        continue

    n_valid   = df["rls_years"].notna().sum()
    n_missing = df["rls_years"].isna().sum()
    frames.append(df)
    print(f"  [OK] {year}: {len(df):,} regencies "
          f"| valid={n_valid}  missing={n_missing} "
          f"| {os.path.basename(path)}")

if not frames:
    raise SystemExit("No RLS files found. Check RLS_DIR path and filenames above.")

if missing_years:
    print(f"\nMissing years: {missing_years}")


# =============================================================================
# STEP 2 -- COMBINE AND CLEAN
# =============================================================================

print("\n" + "=" * 60)
print("STEP 2 -- Combine and clean")
print("=" * 60)

rls_long = pd.concat(frames, ignore_index=True)
rls_long = rls_long.sort_values(["bps_kab_code", "year"]).reset_index(drop=True)

n_reg   = rls_long["bps_kab_code"].nunique()
n_years = rls_long["year"].nunique()
n_obs   = len(rls_long)

print(f"Panel: {n_reg} regencies x {n_years} years = {n_obs:,} obs")
print(f"Years: {sorted(rls_long['year'].unique())}")
print(f"Missing rls_years: {rls_long['rls_years'].isna().sum()}")

year_summary = (
    rls_long.groupby("year")
    .agg(
        regencies = ("bps_kab_code", "nunique"),
        valid     = ("rls_years",    lambda x: x.notna().sum()),
        missing   = ("rls_years",    lambda x: x.isna().sum()),
        mean_rls  = ("rls_years",    "mean"),
        min_rls   = ("rls_years",    "min"),
        max_rls   = ("rls_years",    "max"),
    )
    .reset_index()
)

# Round summary stats
for col in ["mean_rls", "min_rls", "max_rls"]:
    year_summary[col] = year_summary[col].round(2)

print("\nCoverage and summary by year:")
print(year_summary.to_string(index=False))


# =============================================================================
# STEP 3 -- WIDE FORMAT
# =============================================================================

print("\n" + "=" * 60)
print("STEP 3 -- Create rls_wide format")
print("=" * 60)

# One consistent name per regency from most recent year
name_map = (
    rls_long.sort_values("year", ascending=False)
    .drop_duplicates(subset="bps_kab_code")
    [["bps_kab_code", "bps_kab_name", "bps_prov_name", "bps_prov_code"]]
)

rls_wide = rls_long.pivot_table(
    index   = "bps_kab_code",
    columns = "year",
    values  = "rls_years",
    aggfunc = "first",
).reset_index()
rls_wide.columns.name = None
rls_wide.columns = ["bps_kab_code"] + [f"rls_{y}" for y in rls_wide.columns[1:]]
rls_wide = rls_wide.merge(name_map, on="bps_kab_code", how="left")

year_cols = [f"rls_{y}" for y in YEARS if f"rls_{y}" in rls_wide.columns]
rls_wide = rls_wide[[
    "bps_kab_code", "bps_kab_name", "bps_prov_name", "bps_prov_code"
] + year_cols].copy()

print(f"Wide format: {len(rls_wide)} regencies x {len(rls_wide.columns)} columns")
print(rls_wide.head(5).to_string(index=False))


# =============================================================================


# ── Rename to match downstream merge expectations ─────────────────────────
# bps_kab_code in this section is the 4-digit code (built from col[6] of source files).
# Rename to bps_kab_code_4 for consistency with the BPS panel from Section 5,
# which is what the merge in Section 8 joins on.
rls_long = rls_long.rename(columns={"bps_kab_code": "bps_kab_code_4"})
rls_wide = rls_wide.rename(columns={"bps_kab_code": "bps_kab_code_4"})

print(f"\nRLS processed:")
print(f"  Regencies : {rls_long['bps_kab_code_4'].nunique() if 'rls_long' in dir() else 'NA'}")
print(f"  Years     : {sorted(rls_long['year'].unique()) if 'rls_long' in dir() else 'NA'}")
print(f"  Panel obs : {len(rls_long) if 'rls_long' in dir() else 'NA'}")

STEP 1 -- Read annual RLS files

Files in RLS_DIR:
  [Metode Baru] Rata-rata Lama Sekolah, 2016.xlsx
  [Metode Baru] Rata-rata Lama Sekolah, 2017.xlsx
  [Metode Baru] Rata-rata Lama Sekolah, 2018.xlsx
  [Metode Baru] Rata-rata Lama Sekolah, 2019.xlsx
  [Metode Baru] Rata-rata Lama Sekolah, 2020.xlsx
  [Metode Baru] Rata-rata Lama Sekolah, 2021.xlsx
  [Metode Baru] Rata-rata Lama Sekolah, 2022.xlsx
  [Metode Baru] Rata-rata Lama Sekolah, 2023.xlsx
  [Metode Baru] Rata-rata Lama Sekolah, 2024.xlsx
  [Metode Baru] Rata-rata Lama Sekolah, 2025.xlsx

  [OK] 2016: 514 regencies | valid=514  missing=0 | [Metode Baru] Rata-rata Lama Sekolah, 2016.xlsx
  [OK] 2017: 514 regencies | valid=514  missing=0 | [Metode Baru] Rata-rata Lama Sekolah, 2017.xlsx
  [OK] 2018: 514 regencies | valid=514  missing=0 | [Metode Baru] Rata-rata Lama Sekolah, 2018.xlsx
  [OK] 2019: 514 regencies | valid=514  missing=0 | [Metode Baru] Rata-rata Lama Sekolah, 2019.xlsx
  [OK] 2020: 514 regencies | valid=514  missing=

## STEP 8 — Compile full panel

Merge the sector-wide GRDP panel with the poverty and schooling panels into the complete BPS panel.

In [26]:
# ════════════════════════════════════════════════════════════════════════════
# COMPILE FULL PANEL — merge sector wide + poverty + RLS into the BPS panel
# ════════════════════════════════════════════════════════════════════════════

# Build sector wide format from pdrb_panel (from Section 1) — pivot sector_short × year
grdp_pivot = pdrb_panel.pivot_table(
    index   = ["bps_kab_code", "year"],
    columns = "sector_short",
    values  = "value",
    aggfunc = "first",
).reset_index()
grdp_pivot.columns.name = None

# We don't need grdp_total in the wide pivot (already in panel)
if "grdp_total" in grdp_pivot.columns:
    grdp_pivot = grdp_pivot.drop(columns=["grdp_total"])

print(f"GRDP sector wide: {grdp_pivot.shape[1] - 2} sectors × "
      f"{grdp_pivot['bps_kab_code'].nunique()} regencies × "
      f"{grdp_pivot['year'].nunique()} years")

# Merge sector columns into panel (panel from Section 5 already has the 9 base cols)
panel_full = panel.merge(grdp_pivot, on=["bps_kab_code", "year"], how="left")
print(f"After sector merge: {panel_full.shape}")

# Merge poverty (from Section 6)
panel_full = panel_full.merge(
    poverty_long[["bps_kab_code_4", "year", "poverty_pct"]],
    on=["bps_kab_code_4", "year"],
    how="left",
)
print(f"After poverty merge: {panel_full.shape}")

# Merge RLS (from Section 7)
panel_full = panel_full.merge(
    rls_long[["bps_kab_code_4", "year", "rls_years"]],
    on=["bps_kab_code_4", "year"],
    how="left",
)
print(f"After RLS merge: {panel_full.shape}")

# Final column order — base + sectors + welfare
SECTOR_COLS = [c for c in [
    "grdp_A_agriculture", "grdp_B_mining", "grdp_C_manufacturing",
    "grdp_D_electricity", "grdp_E_water", "grdp_F_construction",
    "grdp_G_trade", "grdp_H_transport", "grdp_I_accommodation",
    "grdp_J_infocomm", "grdp_K_finance", "grdp_L_realestate",
    "grdp_MN_business", "grdp_O_government", "grdp_P_education",
    "grdp_Q_health", "grdp_RSTU_other",
] if c in panel_full.columns]

final_cols = (
    ["bps_kab_code", "bps_kab_code_4", "regency_name", "year",
     "grdp_total_billion_idr", "grdp_growth_pct",
     "population", "tpt_pct"]
    + SECTOR_COLS
    + ["poverty_pct", "rls_years"]
)
final_cols = [c for c in final_cols if c in panel_full.columns]
panel_full = panel_full[final_cols].copy()
panel_full = panel_full.sort_values(["bps_kab_code_4", "year"]).reset_index(drop=True)

print(f"\n✓ Full panel assembled")
print(f"  Shape: {panel_full.shape}")
print(f"  Columns: {len(panel_full.columns)} ({len(SECTOR_COLS)} sector + base + welfare)")

# Missing summary
print(f"\nMissing values per column:")
miss_summary = panel_full.isna().sum()
for col, n_miss in miss_summary[miss_summary > 0].sort_values(ascending=False).items():
    print(f"  {col:35s} {n_miss:>5d}  ({n_miss/len(panel_full)*100:.1f}%)")


GRDP sector wide: 17 sectors × 514 regencies × 10 years
After sector merge: (5140, 25)
After poverty merge: (5140, 26)
After RLS merge: (5140, 27)

✓ Full panel assembled
  Shape: (5140, 27)
  Columns: 27 (17 sector + base + welfare)

Missing values per column:
  tpt_pct                               521  (10.1%)
  grdp_growth_pct                       514  (10.0%)


## STEP 9 — Fill missing data

Fill missing unemployment and GRDP-growth values using imputed values, recording every imputed cell in an audit log. Approximately 10% of socioeconomic outcome observations are imputed.

In [35]:
# ══════════════════════════════════════════════════════════════════════
# STEP 1 — LOAD IMPUTATION DATA from missing_data.xlsx
# ══════════════════════════════════════════════════════════════════════
print("\n[1] LOAD IMPUTATION DATA from missing_data.xlsx...")
# Use panel_full from Section 8 instead of reading bps_full_panel.xlsx
panel = panel_full.copy()
panel["bps_kab_code_4"] = pd.to_numeric(
    panel["bps_kab_code_4"], errors="coerce").astype("Int64")
panel["year"] = pd.to_numeric(
    panel["year"], errors="coerce").astype("Int64")

# ── TPT imputation ───────────────────────────────────────────────────
tpt_raw = pd.read_excel(MISS_FILE, sheet_name="tpt_imputation")
tpt_raw.columns = tpt_raw.columns.str.strip()
tpt_raw["bps_kab_code_4"] = pd.to_numeric(
    tpt_raw["bps_kab_code_4"], errors="coerce").astype("Int64")
tpt_raw["year"] = pd.to_numeric(
    tpt_raw["year"], errors="coerce").astype("Int64")
tpt_val_col = [c for c in tpt_raw.columns
               if "tpt" in c.lower() and "code" not in c.lower()][0]
tpt_raw[tpt_val_col] = (
    tpt_raw[tpt_val_col]
    .astype(str).str.replace(",", ".", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)
tpt_imp = (tpt_raw[["bps_kab_code_4", "year", tpt_val_col]]
           .rename(columns={tpt_val_col: "tpt_pct_imp"})
           .copy())

# ── GRDP growth imputation ───────────────────────────────────────────
grdp_raw = pd.read_excel(MISS_FILE, sheet_name="growth_missing")
grdp_raw.columns = grdp_raw.columns.str.strip()
grdp_raw["bps_kab_code_4"] = pd.to_numeric(
    grdp_raw["bps_kab_code_4"], errors="coerce").astype("Int64")
grdp_raw["year"] = pd.to_numeric(
    grdp_raw["year"], errors="coerce").astype("Int64")
grdp_val_col = [c for c in grdp_raw.columns if "growth" in c.lower()][0]
grdp_raw[grdp_val_col] = (
    grdp_raw[grdp_val_col]
    .astype(str).str.replace(",", ".", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)
grdp_imp = (grdp_raw[["bps_kab_code_4", "year", grdp_val_col]]
            .rename(columns={grdp_val_col: "grdp_growth_imp"})
            .copy())

print(f"  Panel          : {panel.shape[0]:,} rows × {panel.shape[1]} cols")
print(f"  TPT imputation : {len(tpt_imp):,} rows  (col='{tpt_val_col}')")
print(f"  GRDP imputation: {len(grdp_imp):,} rows  (col='{grdp_val_col}')")



[1] LOAD IMPUTATION DATA from missing_data.xlsx...
  Panel          : 5,140 rows × 27 cols
  TPT imputation : 522 rows  (col='tpt_pct')
  GRDP imputation: 514 rows  (col='grdp_growth_pct')


In [37]:
# ══════════════════════════════════════════════════════════════════════
# STEP 2 — PRE-FILL AUDIT
# ══════════════════════════════════════════════════════════════════════
print("\n[2] Pre-fill missing data audit...")

def audit(df, label):
    print(f"\n  {label}")
    results = {}
    for col in ["tpt_pct","grdp_growth_pct"]:
        miss = (df.groupby("year")[col]
                .apply(lambda x: x.isna().sum()))
        miss = miss[miss > 0]
        total = int(miss.sum())
        results[col] = {"total": total, "by_year": miss.to_dict()}
        print(f"  {col}: {total} missing total")
        for yr, n in sorted(miss.items()):
            print(f"    year {int(yr)}: {n}")
    return results

pre = audit(panel, "BEFORE imputation:")

# ══════════════════════════════════════════════════════════════════════
# STEP 3 — VALIDATE
# ══════════════════════════════════════════════════════════════════════
print("\n[3] Validating imputation files...")
panel_codes = set(panel["bps_kab_code_4"].dropna().unique())
ok = True

# 3a codes in panel
for name, df in [("TPT", tpt_imp),("GRDP", grdp_imp)]:
    bad = set(df["bps_kab_code_4"].dropna().unique()) - panel_codes
    tag = "OK" if not bad else f"WARN {len(bad)} unknown codes"
    print(f"  {name} codes in panel     : {tag}")
    if bad: print(f"    {sorted(bad)[:10]}")

# 3b NaN in imputed values
for name, df, col in [("TPT", tpt_imp, "tpt_pct_imp"),
                      ("GRDP", grdp_imp, "grdp_growth_imp")
]:
    n = df[col].isna().sum()
    print(f"  {name} NaN in values      : {n}  {'OK' if n==0 else 'ERROR — fix before running'}")
    if n: ok = False

# 3c plausible ranges
tpt_out  = ((tpt_imp["tpt_pct_imp"]      < 0) | (tpt_imp["tpt_pct_imp"]      > 50)).sum()
grdp_out = ((grdp_imp["grdp_growth_imp"] < -50) | (grdp_imp["grdp_growth_imp"] > 100)).sum()
print(f"  TPT  outside [0,50]%    : {tpt_out}   {'OK' if tpt_out==0 else 'WARN'}")
print(f"  GRDP outside [-50,100]% : {grdp_out}  {'OK' if grdp_out==0 else 'WARN'}")

# 3d duplicates
for name, df in [("TPT", tpt_imp),("GRDP", grdp_imp)]:
    d = df.duplicated(subset=["bps_kab_code_4","year"]).sum()
    print(f"  {name} duplicate keys      : {d}  {'OK' if d==0 else 'ERROR'}")
    if d: ok = False

# 3e no overwrite of existing values
for name, pcol, df, icol in [
        ("TPT",  "tpt_pct",        tpt_imp,  "tpt_pct_imp"),
        ("GRDP", "grdp_growth_pct", grdp_imp, "grdp_growth_imp")]:
    chk = panel.merge(df, on=["bps_kab_code_4","year"], how="inner")
    ow  = chk[panel.set_index(["bps_kab_code_4","year"])
              .reindex(chk.set_index(["bps_kab_code_4","year"]).index)[pcol]
              .notna().values].shape[0]
    print(f"  {name} would overwrite     : {ow}  {'OK' if ow==0 else 'WARN — will skip'}")

print(f"\n  Validation result: {'PASSED' if ok else 'ERRORS FOUND — review above'}")

# ══════════════════════════════════════════════════════════════════════
# STEP 4 — MERGE AND FILL
# ══════════════════════════════════════════════════════════════════════
print("\n[4] Filling missing values (NaN-only — never overwrite)...")

# Resolve regency-name column (varies by upstream pipeline)
_name_candidates = ["regency_name_grdp", "regency_name", "bps_kab_name"]
REGENCY_NAME_COL = next((c for c in _name_candidates if c in panel.columns), None)
if REGENCY_NAME_COL is None:
    raise RuntimeError(f"No regency name column found. Tried {_name_candidates}. "
                        f"Available columns: {list(panel.columns)[:20]}")
print(f"  Using regency name column: {REGENCY_NAME_COL}")

panel = panel.merge(tpt_imp,  on=["bps_kab_code_4","year"], how="left")
panel = panel.merge(grdp_imp, on=["bps_kab_code_4","year"], how="left")

n_tpt_fill  = (panel["tpt_pct"].isna()        & panel["tpt_pct_imp"].notna()).sum()
n_grdp_fill = (panel["grdp_growth_pct"].isna() & panel["grdp_growth_imp"].notna()).sum()

# Build audit log before filling
log_rows = []
for _, r in panel[panel["tpt_pct"].isna() & panel["tpt_pct_imp"].notna()].iterrows():
    log_rows.append({
        "bps_kab_code_4": r["bps_kab_code_4"],
        "regency_name":   r[REGENCY_NAME_COL],
        "year":           int(r["year"]),
        "variable":       "tpt_pct",
        "imputed_value":  r["tpt_pct_imp"],
        "method": ("linear interpolation (2015+2017)/2"
                   if int(r["year"]) == 2016
                   else "interpolation or conditional mean (sporadic Papua)"),
    })
for _, r in panel[panel["grdp_growth_pct"].isna() & panel["grdp_growth_imp"].notna()].iterrows():
    log_rows.append({
        "bps_kab_code_4": r["bps_kab_code_4"],
        "regency_name":   r[REGENCY_NAME_COL],
        "year":           int(r["year"]),
        "variable":       "grdp_growth_pct",
        "imputed_value":  r["grdp_growth_imp"],
        "method": "manual collection 2016 GRDP growth",
    })
imp_log = pd.DataFrame(log_rows)

# Fill
panel["tpt_pct"]         = panel["tpt_pct"].fillna(panel["tpt_pct_imp"])
panel["grdp_growth_pct"] = panel["grdp_growth_pct"].fillna(panel["grdp_growth_imp"])
panel = panel.drop(columns=["tpt_pct_imp","grdp_growth_imp"])

print(f"  tpt_pct filled        : {n_tpt_fill}")
print(f"  grdp_growth_pct filled: {n_grdp_fill}")

# ══════════════════════════════════════════════════════════════════════
# STEP 5 — POST-FILL AUDIT
# ══════════════════════════════════════════════════════════════════════
print("\n[5] Post-fill audit...")
post = audit(panel, "AFTER imputation:")

print("\n  Fill summary:")
for col in ["tpt_pct","grdp_growth_pct"]:
    b = pre[col]["total"]; a = post[col]["total"]
    print(f"  {col:<25}: {b} -> {a} missing  ({b-a} filled)")

# ══════════════════════════════════════════════════════════════════════
# STEP 6 — STATISTICAL DESCRIPTION
# ══════════════════════════════════════════════════════════════════════
print("\n[6] Statistical description — imputed vs observed years...")

for col, label in [("tpt_pct","TPT (%)"),
                   ("grdp_growth_pct","GRDP Growth (%)")]:
    print(f"\n  {label} by year:")
    print(f"  {'Year':<6} {'N':>5} {'Mean':>8} {'Median':>8} "
          f"{'Std':>7} {'Min':>8} {'Max':>8}")
    print("  " + "-"*55)
    for yr in sorted(panel["year"].unique()):
        s   = panel[panel["year"]==yr][col].dropna()
        tag = " <- imputed" if yr == 2016 else ""
        print(f"  {int(yr):<6} {len(s):>5} {s.mean():>8.3f} {s.median():>8.3f} "
              f"{s.std():>7.3f} {s.min():>8.3f} {s.max():>8.3f}{tag}")

    # Detailed stats for 2016
    s2016 = panel[panel["year"]==2016][col].dropna()
    print(f"\n  {label} 2016 detail (n={len(s2016)}):")
    for k,v in s2016.describe().round(4).items():
        print(f"    {k:<8}: {v:.4f}")
    print(f"    skewness : {s2016.skew():.4f}")
    print(f"    kurtosis : {s2016.kurtosis():.4f}")
    neg = (s2016 < 0).sum()
    if neg:
        print(f"    negative : {neg} (economic contraction — verify data source)")

# ══════════════════════════════════════════════════════════════════════
# STEP 7 — CONSISTENCY CHECKS
# ══════════════════════════════════════════════════════════════════════
print("\n[7] Consistency checks...")

# A: TPT 2016 national mean
tpt16 = panel[panel["year"]==2016]["tpt_pct"].mean()
tpt17 = panel[panel["year"]==2017]["tpt_pct"].mean()
print(f"\n  A. TPT national mean:  2016={tpt16:.3f}%  2017={tpt17:.3f}%")
print(f"     {'OK — plausible' if 0 < tpt16 < 30 else 'WARN — check values'}")

# B: GRDP growth 2016 vs all years
grdp16  = panel[panel["year"]==2016]["grdp_growth_pct"].mean()
grdp_mu = panel["grdp_growth_pct"].mean()
print(f"\n  B. GRDP growth:  2016 mean={grdp16:.3f}%  all-years mean={grdp_mu:.3f}%")
print(f"     {'OK' if -10 < grdp16 < 30 else 'WARN — unexpected range'}")

# C: Outliers in GRDP 2016 (>3 sigma)
g16 = panel[panel["year"]==2016]["grdp_growth_pct"].dropna()
mu, sd = g16.mean(), g16.std()
ol = panel[(panel["year"]==2016) &
           ((panel["grdp_growth_pct"]>mu+3*sd)|
            (panel["grdp_growth_pct"]<mu-3*sd))][
    ["bps_kab_code_4", REGENCY_NAME_COL, "grdp_growth_pct"]]
print(f"\n  C. GRDP 2016 outliers (>3s):  {len(ol)}")
print(f"     range 3s: [{mu-3*sd:.2f}, {mu+3*sd:.2f}]")
if len(ol):
    print(ol.to_string(index=False))

# D: Panel balance
nc = panel["bps_kab_code_4"].nunique()
ny = panel["year"].nunique()
print(f"\n  D. Panel balance: {nc} regencies x {ny} years = {nc*ny:,} expected")
print(f"     Actual rows: {len(panel):,}  "
      f"{'OK balanced' if len(panel)==nc*ny else 'WARN unbalanced'}")

# E: Final NaN counts
print(f"\n  E. Final NaN in key columns:")
for col in ["tpt_pct","grdp_growth_pct","poverty_pct","rls_years","population"]:
    n = panel[col].isna().sum()
    print(f"     {col:<25}: {n}")

# ══════════════════════════════════════════════════════════════════════
# STEP 8 — ALL VARIABLES MISSING AUDIT
# ══════════════════════════════════════════════════════════════════════
print("\n[8] Final complete missing audit (all variables)...")
print(f"\n  {'Variable':<35} {'Missing':>8} {'Pct':>8}")
print("  " + "-"*54)
skip = {"bps_kab_code_4", "bps_kab_code", "bps_kab_code_7", REGENCY_NAME_COL, "year"}
for col in panel.columns:
    if col in skip: continue
    n   = panel[col].isna().sum()
    pct = n / len(panel) * 100
    flag = " <-" if n > 0 else ""
    print(f"  {col:<35} {n:>8,} {pct:>7.3f}%{flag}")

# ══════════════════════════════════════════════════════════════════════


[2] Pre-fill missing data audit...

  BEFORE imputation:
  tpt_pct: 521 missing total
    year 2016: 514
    year 2017: 2
    year 2019: 2
    year 2021: 2
    year 2024: 1
  grdp_growth_pct: 514 missing total
    year 2016: 514

[3] Validating imputation files...
  TPT codes in panel     : OK
  GRDP codes in panel     : OK
  TPT NaN in values      : 0  OK
  GRDP NaN in values      : 0  OK
  TPT  outside [0,50]%    : 0   OK
  GRDP outside [-50,100]% : 0  OK
  TPT duplicate keys      : 0  OK
  GRDP duplicate keys      : 0  OK
  TPT would overwrite     : 1  WARN — will skip
  GRDP would overwrite     : 0  OK

  Validation result: PASSED

[4] Filling missing values (NaN-only — never overwrite)...
  Using regency name column: regency_name
  tpt_pct filled        : 521
  grdp_growth_pct filled: 514

[5] Post-fill audit...

  AFTER imputation:
  tpt_pct: 0 missing total
  grdp_growth_pct: 0 missing total

  Fill summary:
  tpt_pct                  : 521 -> 0 missing  (521 filled)
  grdp_gro

## STEP 10 — Export final panel

Write the final panel with three sheets: the full regency × year × variable panel, the imputation audit log, and the data dictionary.

**Output:** `data/processed/3_bps_full_panel_complete.xlsx`

In [40]:
# ════════════════════════════════════════════════════════════════════════════
# FINAL EXPORT — bps_full_panel_complete.xlsx
# Single file with three styled sheets
# ════════════════════════════════════════════════════════════════════════════

print(f"\n{'=' * 65}")
print(f"FINAL EXPORT → {OUTPUT_PATH}")
print(f"{'=' * 65}")

# Build imputation log if it doesn't exist (defensive — nb7 creates it)
if "imp_log" not in dir():
    imp_log = pd.DataFrame(columns=["bps_kab_code_4", "year", "variable", "imputed_value", "source"])

# Build data dictionary
data_dict_rows = []

# Identification columns
data_dict_rows += [
    {"variable": "bps_kab_code",     "type": "string",  "description": "7-digit BPS regency code"},
    {"variable": "bps_kab_code_4",   "type": "Int64",   "description": "4-digit BPS regency code (join key)"},
    {"variable": "regency_name",     "type": "string",  "description": "Regency / city name"},
    {"variable": "year",             "type": "Int64",   "description": "Calendar year (2016–2025)"},
]
# Core economic columns
data_dict_rows += [
    {"variable": "grdp_total_billion_idr",    "type": "float",
     "description": "GRDP at constant 2010 prices (billion IDR)"},
    {"variable": "grdp_growth_pct",           "type": "float",
     "description": "Annual GRDP growth rate (%) — 2016 imputed via missing_data.xlsx"},
    {"variable": "population",                "type": "Int64",
     "description": "Total population (persons)"},
    {"variable": "tpt_pct",                   "type": "float",
     "description": "Open unemployment rate (%) — 2016 imputed via missing_data.xlsx"},
]
# Sector columns
SECTOR_DESCRIPTIONS = {
    "grdp_A_agriculture":    "GRDP, sector A: Agriculture, Forestry, Fisheries (billion IDR)",
    "grdp_B_mining":         "GRDP, sector B: Mining and Quarrying (billion IDR)",
    "grdp_C_manufacturing":  "GRDP, sector C: Manufacturing (billion IDR)",
    "grdp_D_electricity":    "GRDP, sector D: Electricity and Gas (billion IDR)",
    "grdp_E_water":          "GRDP, sector E: Water, Waste, Recycling (billion IDR)",
    "grdp_F_construction":   "GRDP, sector F: Construction (billion IDR)",
    "grdp_G_trade":          "GRDP, sector G: Wholesale and Retail Trade (billion IDR)",
    "grdp_H_transport":      "GRDP, sector H: Transport and Warehousing (billion IDR)",
    "grdp_I_accommodation":  "GRDP, sector I: Accommodation and Food Service (billion IDR)",
    "grdp_J_infocomm":       "GRDP, sector J: Information and Communication (billion IDR)",
    "grdp_K_finance":        "GRDP, sector K: Financial and Insurance Services (billion IDR)",
    "grdp_L_realestate":     "GRDP, sector L: Real Estate (billion IDR)",
    "grdp_MN_business":      "GRDP, sectors M+N: Business Services (billion IDR)",
    "grdp_O_government":     "GRDP, sector O: Public Administration (billion IDR)",
    "grdp_P_education":      "GRDP, sector P: Education Services (billion IDR)",
    "grdp_Q_health":         "GRDP, sector Q: Health Services (billion IDR)",
    "grdp_RSTU_other":       "GRDP, sectors R+S+T+U: Other Services (billion IDR)",
}
for col, desc in SECTOR_DESCRIPTIONS.items():
    if col in panel.columns:
        data_dict_rows.append({"variable": col, "type": "float", "description": desc})

# Welfare columns
data_dict_rows += [
    {"variable": "poverty_pct", "type": "float",
     "description": "Poverty rate P0 (%) — % population below poverty line"},
    {"variable": "rls_years",   "type": "float",
     "description": "Mean years of schooling (RLS) — population aged 15+"},
]

data_dictionary = pd.DataFrame(data_dict_rows)
print(f"  Data dictionary: {len(data_dictionary)} variables")

# Build the workbook
wb = openpyxl.Workbook()
wb.remove(wb.active)

# Sheet 1: Panel (the main data)
ws1 = wb.create_sheet("Panel")
write_sheet(ws1, panel,
            title=f"BPS Full Panel — {panel['bps_kab_code_4'].nunique()} regencies × "
                  f"{panel['year'].nunique()} years = {len(panel):,} obs")

# Sheet 2: Imputation log
ws2 = wb.create_sheet("Imputation_Log")
write_sheet(ws2, imp_log,
            title=f"Imputation log — {len(imp_log):,} cells filled from missing_data.xlsx")

# Sheet 3: Data dictionary
ws3 = wb.create_sheet("Data_Dictionary")
write_sheet(ws3, data_dictionary,
            title="BPS Full Panel — variable descriptions")

wb.save(OUTPUT_PATH)

import os
size_kb = os.path.getsize(OUTPUT_PATH) / 1024
print(f"\n  ✓ {OUTPUT_PATH}: {size_kb:.0f} KB")
print(f"    Sheets: {wb.sheetnames}")
print(f"\n{'=' * 65}")
print("DONE — bps_full_panel_complete.xlsx ready for downstream notebooks")
print(f"{'=' * 65}")



FINAL EXPORT → ../data/processed/3_bps_full_panel_complete.xlsx
  Data dictionary: 27 variables

  ✓ ../data/processed/3_bps_full_panel_complete.xlsx: 1542 KB
    Sheets: ['Panel', 'Imputation_Log', 'Data_Dictionary']

DONE — bps_full_panel_complete.xlsx ready for downstream notebooks


## Summary

| File | Sheets | Contents |
|------|--------|----------|
| `3_bps_full_panel_complete.xlsx` | Panel · Imputation_Log · Data_Dictionary | Final BPS panel ready for the RQ1 / RQ2 analysis |

All intermediate tables are held in memory rather than written to disk.

**Next step:** Notebook 12 reads `3_bps_full_panel_complete.xlsx` and merges it with the BNPB flood data to produce `rq2_master_panel.csv`.